# Case Study 2: Hierarchical Linear Model (HLM)

**Circulatory Fidelity: Quantifying Structural Coupling to Diagnose Mean-Field Failure**

This notebook demonstrates how CF diagnoses no-pooling failure in hierarchical models.

---

## Model Specification

$$
\begin{align}
\theta_j &\sim \mathcal{N}(0, \tau^2), \quad j = 1, \ldots, J \quad \text{[group effects]}\\
y_{ij} | \theta_j &\sim \mathcal{N}(\theta_j, \sigma^2), \quad i = 1, \ldots, n \quad \text{[observations]}
\end{align}
$$

Key quantities:
- **ICC**: $\frac{\tau^2}{\tau^2 + \sigma^2}$ â€” fraction of variance between groups
- **Reliability**: $\frac{\tau^2}{\tau^2 + \sigma^2/n}$ â€” how well group means estimate true effects

For HLM, **CF equals reliability** by construction.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from dataclasses import dataclass
from typing import NamedTuple

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 11
np.random.seed(42)

## HLM Model Implementation

In [ ]:
@dataclass
class HLMParams:
    """HLM model parameters (matching manuscript)."""
    n_groups: int = 30      # J: number of groups
    n_per_group: int = 10   # n: observations per group
    tau: float = 1.0        # Ï„: between-group SD (signal)
    sigma: float = 1.0      # Ïƒ: within-group SD (noise)
    mu: float = 0.0         # Î¼: grand mean
    
    @property
    def icc(self) -> float:
        """Intraclass correlation coefficient."""
        return self.tau**2 / (self.tau**2 + self.sigma**2)
    
    @property
    def reliability(self) -> float:
        """Reliability of group means = CF for HLM."""
        return self.tau**2 / (self.tau**2 + self.sigma**2 / self.n_per_group)

class HLMSimulation(NamedTuple):
    theta: np.ndarray
    y: np.ndarray
    y_bar: np.ndarray
    params: HLMParams

def simulate_hlm(params: HLMParams, seed: int = None) -> HLMSimulation:
    """Simulate from HLM generative model."""
    if seed is not None:
        np.random.seed(seed)
    
    theta = np.random.normal(params.mu, params.tau, params.n_groups)
    y = np.zeros((params.n_groups, params.n_per_group))
    
    for j in range(params.n_groups):
        y[j] = np.random.normal(theta[j], params.sigma, params.n_per_group)
    
    y_bar = y.mean(axis=1)
    return HLMSimulation(theta=theta, y=y, y_bar=y_bar, params=params)

## CF Computation for HLM

For HLM, CF = reliability by construction:
$$\text{CF} = \frac{\tau^2}{\tau^2 + \sigma^2/n}$$

In [ ]:
def compute_cf_hlm(params: HLMParams) -> float:
    """
    CF for HLM equals reliability.
    Low CF = weak signal = no-pooling will overfit.
    """
    return params.reliability

## Visualization: The Pooling Problem

In [ ]:
# Low signal scenario (low Ï„ â†’ low CF)
params_low = HLMParams(tau=0.3, sigma=1.0)
sim_low = simulate_hlm(params_low, seed=42)
cf_low = compute_cf_hlm(params_low)

# High signal scenario (high Ï„ â†’ high CF)
params_high = HLMParams(tau=2.0, sigma=1.0)
sim_high = simulate_hlm(params_high, seed=42)
cf_high = compute_cf_hlm(params_high)

print("Low Signal Scenario:")
print(f"  Ï„ = {params_low.tau}, ICC = {params_low.icc:.3f}, CF = {cf_low:.3f}")
print()
print("High Signal Scenario:")
print(f"  Ï„ = {params_high.tau}, ICC = {params_high.icc:.3f}, CF = {cf_high:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, sim, params, cf, title in [
    (axes[0], sim_low, params_low, cf_low, 'Low Signal (Low CF)'),
    (axes[1], sim_high, params_high, cf_high, 'High Signal (High CF)')
]:
    ax.scatter(sim.theta, sim.y_bar, alpha=0.7, s=50, c='black')
    ax.plot([-4, 4], [-4, 4], 'k--', alpha=0.3, label='Perfect estimation')
    ax.set_xlabel('True Group Effect Î¸')
    ax.set_ylabel('Group Mean $\\bar{y}_j$')
    ax.set_title(f'{title}\nÏ„={params.tau}, CF={cf:.2f}')
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.legend()

plt.tight_layout()
plt.show()

## Inference Methods: No-Pooling vs Partial Pooling

In [ ]:
def no_pooling_estimator(sim: HLMSimulation):
    """No-pooling: Use group means directly (mean-field equivalent)."""
    theta_hat = sim.y_bar
    mse = np.mean((theta_hat - sim.theta)**2)
    return theta_hat, mse

def partial_pooling_estimator(sim: HLMSimulation):
    """Partial pooling: Shrink toward grand mean using reliability."""
    grand_mean = np.mean(sim.y_bar)
    shrinkage = sim.params.reliability
    theta_hat = shrinkage * sim.y_bar + (1 - shrinkage) * grand_mean
    mse = np.mean((theta_hat - sim.theta)**2)
    return theta_hat, mse

In [ ]:
print(f"LOW SIGNAL (CF = {cf_low:.2f}):")
np_est, np_mse = no_pooling_estimator(sim_low)
pp_est, pp_mse = partial_pooling_estimator(sim_low)
print(f"  No-pooling MSE:      {np_mse:.4f}")
print(f"  Partial-pooling MSE: {pp_mse:.4f}")
print(f"  MSE Ratio:           {np_mse/pp_mse:.2f}x")
print()
print(f"HIGH SIGNAL (CF = {cf_high:.2f}):")
np_est, np_mse = no_pooling_estimator(sim_high)
pp_est, pp_mse = partial_pooling_estimator(sim_high)
print(f"  No-pooling MSE:      {np_mse:.4f}")
print(f"  Partial-pooling MSE: {pp_mse:.4f}")
print(f"  MSE Ratio:           {np_mse/pp_mse:.2f}x")

## Parameter Sweep: CF vs Inference Performance

In [ ]:
def run_hlm_sweep(tau_values, n_sims=100):
    results = []
    for tau in tau_values:
        params = HLMParams(tau=tau, sigma=1.0)
        cf = compute_cf_hlm(params)
        for rep in range(n_sims):
            sim = simulate_hlm(params)
            _, np_mse = no_pooling_estimator(sim)
            _, pp_mse = partial_pooling_estimator(sim)
            results.append({
                'tau': tau, 'icc': params.icc, 'cf': cf,
                'no_pool_mse': np_mse, 'partial_pool_mse': pp_mse,
                'mse_ratio': np_mse / max(pp_mse, 1e-10)
            })
    return pd.DataFrame(results)

tau_values = [0.2, 0.4, 0.6, 0.8, 1.0, 1.5, 2.0, 3.0]
results = run_hlm_sweep(tau_values, n_sims=50)
summary = results.groupby('tau').agg({'cf': 'mean', 'icc': 'mean', 'mse_ratio': ['mean', 'std']}).round(3)
print(summary)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

cf_by_tau = results.groupby('tau')['cf'].mean()
axes[0].plot(cf_by_tau.index, cf_by_tau.values, 'ko-', markersize=8)
axes[0].axhline(0.4, color='red', linestyle='--', label='Threshold (0.4)')
axes[0].set_xlabel('Between-group SD (Ï„)')
axes[0].set_ylabel('CF')
axes[0].set_title('A. CF increases with signal strength')
axes[0].legend()

for tau in tau_values:
    data = results[results['tau'] == tau]['mse_ratio']
    axes[1].boxplot(data, positions=[tau], widths=0.15)
axes[1].set_xlabel('Between-group SD (Ï„)')
axes[1].set_ylabel('MSE Ratio (No-Pool / Partial-Pool)')
axes[1].set_title('B. No-pooling fails at low Ï„')

axes[2].scatter(results['cf'], results['mse_ratio'], alpha=0.3, s=20, c='black')
r, p = stats.pearsonr(results['cf'], results['mse_ratio'])
axes[2].set_xlabel('CF')
axes[2].set_ylabel('MSE Ratio')
axes[2].set_title(f'C. Low CF predicts no-pooling failure (r={r:.2f})')
axes[2].axvline(0.4, color='red', linestyle='--')

plt.tight_layout()
plt.show()

## Key Findings

1. **CF = reliability** for HLM: Low Ï„ â†’ Low reliability â†’ Low CF
2. **Low CF predicts no-pooling failure**: When CF < 0.4, shrinkage helps
3. **Inverse relationship**: Unlike SVF, here *low* CF indicates problems

### Practical Recommendation

For HLM-like pooling models:
- **CF < 0.4**: Use partial pooling (hierarchical shrinkage)
- **CF > 0.4**: No-pooling acceptable (groups well-differentiated)